# MERO AI ROBOT - YOLOv8 Training
Runtime > Change runtime type > T4 GPU

In [ ]:
# GPU 확인
!nvidia-smi

In [ ]:
# 패키지 설치
!pip install ultralytics roboflow -q

In [ ]:
# Google Drive 연결 (결과 저장용 - 세션 끊겨도 안 날아감)
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/MERO_AI_ROBOT'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 경로: {SAVE_DIR}')

In [ ]:
# Roboflow에서 데이터셋 다운로드
# Roboflow > 프로젝트 > Versions > Export > Show Download Code 에서 확인
from roboflow import Roboflow

API_KEY   = "YOUR_API_KEY"       # Roboflow Settings > API Key
WORKSPACE = "YOUR_WORKSPACE"
PROJECT   = "YOUR_PROJECT_NAME"
VERSION   = 1

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(f'데이터셋 경로: {dataset.location}')

In [ ]:
# 클래스 및 이미지 수 확인
import yaml

with open(f'{dataset.location}/data.yaml', 'r') as f:
    info = yaml.safe_load(f)

print('클래스:', info['names'])
print('클래스 수:', info['nc'])

for split in ['train', 'valid', 'test']:
    img_dir = f"{dataset.location}/{split}/images"
    if os.path.exists(img_dir):
        print(f'{split}: {len(os.listdir(img_dir))}장')

In [ ]:
# YOLOv8 학습
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # s=small: 속도와 정확도 균형 (대회용 추천)

model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,       # 20 에폭 동안 성능 향상 없으면 자동 종료
    lr0=0.01,
    augment=True,
    hsv_h=0.015,       # 색상 변화 증강
    hsv_s=0.7,         # 채도 변화 증강
    hsv_v=0.4,         # 밝기 변화 증강 (조명 차이 대응)
    fliplr=0.5,        # 좌우 반전
    degrees=45,        # 회전 증강 (주사위 방향 무관하게 대응)
    scale=0.5,         # 크기 변화 증강 (카메라 거리 차이 대응)
    mosaic=1.0,        # 모자이크 증강 (여러 물체 동시 탐지)
    project=SAVE_DIR,
    name='train',
    exist_ok=True,
)

In [ ]:
# 학습 결과 확인
from IPython.display import Image, display

result_dir = f'{SAVE_DIR}/train'

# 학습 곡선 (loss, mAP 추이)
display(Image(filename=f'{result_dir}/results.png', width=900))

In [ ]:
# Confusion Matrix (어떤 클래스를 헷갈리는지 확인)
from IPython.display import Image, display

result_dir = f'{SAVE_DIR}/train'
display(Image(filename=f'{result_dir}/confusion_matrix_normalized.png', width=700))

In [ ]:
# 최종 mAP 수치 출력
from ultralytics import YOLO

result_dir = f'{SAVE_DIR}/train'
best_model = YOLO(f'{result_dir}/weights/best.pt')
metrics = best_model.val(data=f'{dataset.location}/data.yaml', verbose=False)

print(f'mAP50    : {metrics.box.map50:.4f}')   # 주요 지표
print(f'mAP50-95 : {metrics.box.map:.4f}')     # 엄격한 지표

# 클래스별 성능
for i, name in enumerate(info['names']):
    print(f'  {name}: AP50 = {metrics.box.ap50[i]:.4f}')

In [ ]:
# best.pt 로컬로 다운로드
from google.colab import files

result_dir = f'{SAVE_DIR}/train'
files.download(f'{result_dir}/weights/best.pt')